In [2]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv
import os
from langchain_core.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
load_dotenv(override=True)
 

/home/ubuntu/My_learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
llm = ChatAnthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    model=os.environ.get("CLAUDE_MODEL"),             # Required: model ID to use
    temperature=1.0,                       # Randomness: 0.0 = deterministic, 1.0 = default (max varies by model)
    max_tokens=256,                       # Max tokens in response (default from model profile, fallback 4096)
    top_k=None,                            # Top-K sampling: consider only top K tokens (None = disabled)
    top_p=None,                            # Top-P sampling: nucleus sampling threshold (None = disabled)
    timeout=None,                          # Request timeout in seconds (None = no timeout)
    max_retries=2,                         # Number of retries on failed requests
    stop=None,                             # Stop sequences e.g. ["\n", "END"] (None = disabled)
    streaming=True,                       # Stream tokens as generated (False = wait for full response)
    thinking=None,
    base_url=os.environ.get("ENDPOINT") 
)

In [5]:
class PipelineState(TypedDict):
    text: str
    char_count: int
    word_count: int
    summary: str


# 2) Existing count node (character count)
def count_chars(state: PipelineState) -> dict:
    return {"char_count": len(state["text"])}


# 3) New node: word_count
def count_words(state: PipelineState) -> dict:
    words = state["text"].split()
    return {"word_count": len(words)}


# 4) Update summarise node to include both counts
def make_summary(state: PipelineState) -> dict:
    text = state["text"]
    chars = state["char_count"]
    words = state["word_count"]
    return {"summary": f"Summary: '{text}' — {chars} characters, {words} words."}


# Build graph
graph_builder = StateGraph(PipelineState)

graph_builder.add_node("count", count_chars)
graph_builder.add_node("word_count", count_words)
graph_builder.add_node("summarise", make_summary)

# START → count → word_count → summarise → END
graph_builder.add_edge(START, "count")
graph_builder.add_edge("count", "word_count")
graph_builder.add_edge("word_count", "summarise")
graph_builder.add_edge("summarise", END)

graph = graph_builder.compile()

# Test input
result = graph.invoke({"text": "As you can see I'm not dead yet so there is no question yielding!"})
print(result["summary"])

Summary: 'As you can see I'm not dead yet so there is no question yielding!' — 65 characters, 14 words.
